# Lie Groups and Lie Algebras

A hands-on introduction to the algebraic and geometric structures used to represent rigid-body motion, rotation, and similarity transformations in robotics, computer vision, and SLAM. The notebook is structured as a self-contained progression from elementary calculus to fully worked examples on $SE(3)$.

**Reading guide.** Sections build on each other. If you only need a refresher, jump straight to the section you want — every chapter recaps any prerequisite formula it uses.

**Companion notebooks.** For SLAM-focused minimal preliminaries, see [`lie_groups_se2.ipynb`](lie_groups_se2.ipynb). For pose-graph and factor-graph applications, see [`pose_graph_slam.ipynb`](pose_graph_slam.ipynb) and [`factor_graph.ipynb`](factor_graph.ipynb).

---

**Outline:**

1. **Mathematical preliminaries** — Taylor's theorem, Taylor series, matrix exponential, Euler's formula.
2. **Groups** — group axioms, matrix groups ($GL,\ SL,\ \mathrm{Aff},\ O,\ SO,\ SE$).
3. **Manifolds and tangent spaces** — what they are, why matrix groups are *also* geometric objects.
4. **Lie groups** — group + smooth manifold; how the constraint $R^TR = I$ produces a tangent space of skew-symmetric matrices.
5. **Lie algebras** — abstract definition; tangent space at identity; generators; hat / vee notation; the cross product as a Lie bracket.
6. **The exponential and logarithm maps** — closed-form derivations for $SO(2),\ SO(3)$ (Rodrigues), $SE(2),\ SE(3)$.
7. **Practical operations** — action on a point, $\oplus$ / $\ominus$ operators, adjoint representation, basis change.
8. **Calculus on Lie groups** — Jacobians, derivatives of group actions.
9. **Quaternions** — definition, multiplication as a $4\times 4$ matrix product, conversion to rotation matrices.
10. **Generalisations beyond $SE(3)$** — $\mathrm{Sim}(2),\ \mathrm{Sim}(3),\ SE_2(3)$ (used in IMU-aided SLAM).
11. **Applications in robotics** — angular/linear velocity, screws, twists, pose representation, integration over $SO(3)$, EKF on the manifold.
12. **Reference cheat sheet** — a single table summarising every Lie group used in robotics.

---


## 1. Mathematical Preliminaries

We need two mathematical tools throughout: **Taylor expansion** (to derive closed forms for matrix exponentials) and **Euler's formula** (the simplest non-trivial example of an exponential map onto a Lie group, namely $S^1 \subset \mathbb{C}^*$).

If both are familiar, skim and move on.


### 1.1 Taylor's theorem

**Single-variable form.** For a sufficiently smooth function $f$ near a point $a$:

$$
f(x) = f(a) + f'(a)(x-a) + \tfrac{1}{2!}f''(a)(x-a)^2 + \tfrac{1}{3!}f'''(a)(x-a)^3 + \cdots
\;=\; \sum_{j=0}^{\infty} \frac{f^{(j)}(a)}{j!}(x-a)^j
$$

**Multivariable form.** For $f:\mathbb{R}^n\to\mathbb{R}$:

$$
f(\mathbf{x}) \approx f(\mathbf{a}) + Df(\mathbf{a})_{1\times n}\,(\mathbf{x}-\mathbf{a})_{n\times 1}
+ \tfrac12\, (\mathbf{x}-\mathbf{a})^T_{1\times n}\, Hf(\mathbf{a})_{n\times n}\, (\mathbf{x}-\mathbf{a})_{n\times 1}
$$

where $Df$ is the gradient row-vector and $Hf$ is the Hessian matrix.


### 1.2 Taylor series for sine, cosine, exponential

Centred at $x = 0$, three series we will reuse repeatedly when computing matrix exponentials.

**Sine.** Derivatives at $0$ cycle through $\cos 0 = 1,\ -\sin 0 = 0,\ -\cos 0 = -1,\ \sin 0 = 0$, so only odd powers of $x$ remain:

$$
\sin x = x - \frac{x^3}{3!} + \frac{x^5}{5!} - \frac{x^7}{7!} + \cdots
\;=\; \sum_{n=0}^{\infty} \frac{(-1)^n}{(2n+1)!}\, x^{2n+1}
$$

**Cosine.** Similarly, only even powers:

$$
\cos x = 1 - \frac{x^2}{2!} + \frac{x^4}{4!} - \frac{x^6}{6!} + \cdots
\;=\; \sum_{n=0}^{\infty} \frac{(-1)^n}{(2n)!}\, x^{2n}
$$

**Exponential.** Every derivative of $e^x$ is $e^x$ itself and $e^0 = 1$, so

$$
e^x = 1 + x + \frac{x^2}{2!} + \frac{x^3}{3!} + \cdots
\;=\; \sum_{k=0}^{\infty} \frac{x^k}{k!}.
$$

This series converges for *every* real $x$ — and, as we will see, for every square matrix $X$ as well.


### 1.3 The matrix exponential

The same Taylor series defines the exponential of a square matrix:

$$
\boxed{\;
e^{\mathbf{X}} \;=\; \sum_{k=0}^{\infty} \frac{1}{k!}\, \mathbf{X}^k
\;=\; \mathbf{I} + \mathbf{X} + \tfrac12 \mathbf{X}^2 + \tfrac{1}{3!}\mathbf{X}^3 + \cdots
\;}
$$

This series converges absolutely for every square matrix $\mathbf{X}$. Two important properties:

* $e^{\mathbf{0}} = \mathbf{I}$.
* $e^{\mathbf{X}}\, e^{-\mathbf{X}} = \mathbf{I}$, so the matrix exponential is always invertible.

**Warning.** $e^{\mathbf{X}+\mathbf{Y}} = e^{\mathbf{X}}\, e^{\mathbf{Y}}$ holds *only when* $\mathbf{X}$ and $\mathbf{Y}$ commute. This non-commutativity is exactly what makes Lie theory non-trivial — the Baker–Campbell–Hausdorff formula corrects for it.


### 1.4 Euler's formula

Apply the matrix-exponential definition to the scalar $i\theta$:

$$
\begin{aligned}
e^{i\theta} &= 1 + i\theta + \frac{(i\theta)^2}{2!} + \frac{(i\theta)^3}{3!} + \frac{(i\theta)^4}{4!} + \cdots \\
&= 1 + i\theta - \frac{\theta^2}{2!} - \frac{i\theta^3}{3!} + \frac{\theta^4}{4!} + \frac{i\theta^5}{5!} - \cdots \\
&= \underbrace{\Big(1 - \tfrac{\theta^2}{2!} + \tfrac{\theta^4}{4!} - \cdots \Big)}_{\cos\theta}
   + i\,\underbrace{\Big(\theta - \tfrac{\theta^3}{3!} + \tfrac{\theta^5}{5!} - \cdots \Big)}_{\sin\theta}.
\end{aligned}
$$

So **Euler's formula** is:

$$
\boxed{\; e^{i\theta} = \cos\theta + i\sin\theta \;}
$$

Geometrically: the exponential map sends a real number $\theta$ (an element of the tangent line at $1$ in the unit circle $S^1 \subset \mathbb{C}^*$) to the point $e^{i\theta}$ on the unit circle. This is the simplest non-trivial example of a Lie-group exponential map — and exactly the same construction we will use for $SO(2),\ SO(3),\ SE(3)$ later. **Keep this picture in mind.**


## 2. Groups

A **group** is the algebraic abstraction we impose on the set of allowable transformations (rotations, rigid motions, similarity transforms). The four group axioms guarantee that compositions and inverses behave the way we intuitively expect.


### 2.1 Definition of a group

Let $G$ be a set and $\cdot$ a binary operation $\cdot: G \times G \to G$. Then $(G, \cdot)$ is a **group** if all four axioms hold:

1. **Closure.** For every pair $a, b \in G$, the product $a \cdot b \in G$.
2. **Associativity.** $\forall a, b, c \in G,\ (a \cdot b) \cdot c = a \cdot (b \cdot c)$.
3. **Identity element.** $\exists\, e \in G$ such that $\forall a \in G,\ e \cdot a = a \cdot e = a$.
4. **Inverse element.** $\forall a \in G,\ \exists\, b \in G$ such that $a \cdot b = b \cdot a = e$.

If $a \cdot b = b \cdot a$ for all $a, b$, the group is **abelian** (commutative). Most rotation/rigid-motion groups in robotics are *not* abelian — composing rotations does depend on order.


### 2.2 Examples of groups

| Group | Set | Operation | Identity | Inverse of $a$ | Abelian? |
|---|---|---|---|---|---|
| $(\mathbb{Z}, +)$ | integers | addition | $0$ | $-a$ | yes |
| $(\mathbb{R}^*, \times)$ | non-zero reals | multiplication | $1$ | $1/a$ | yes |
| $(GL(n,\mathbb{R}),\, \cdot)$ | invertible $n\times n$ matrices | matrix multiplication | $I_n$ | $A^{-1}$ | **no** (in general) |

The first two are textbook examples. The third — invertible matrices under multiplication — is the gateway to every group we actually use in robotics.


### 2.3 Matrix groups

**Matrix groups** (also called linear groups) are sets of matrices closed under matrix multiplication and inversion. They are subgroups of the **general linear group** $GL(n, F)$, where $n$ is the matrix size and $F$ is the field (typically $\mathbb{R}$ or $\mathbb{C}$).

For a set of matrices to form a matrix group:

1. **Closure** — the product of any two matrices in the set is in the set.
2. **Associativity** — matrix multiplication is associative.
3. **Identity** — the set contains $I_n$.
4. **Inverse** — every matrix in the set has its inverse in the set.

Each matrix group is a *constrained* subset of $n^2$-dimensional Euclidean space. The constraints reduce the degrees of freedom — e.g., $SO(3)$ has 3 DoF even though its elements are $3\times 3$ matrices with 9 entries.


### 2.4 Important matrix groups

The matrix groups we will return to repeatedly. Each is a subgroup of $GL(n,\mathbb{R})$ defined by an additional algebraic constraint.


#### 2.4.1 General linear group $GL(n, \mathbb{R})$

The set of all invertible $n \times n$ real matrices, with matrix multiplication.

* Identity: $I_n$.
* Inverse of $A$: $A^{-1}$ (exists by definition of invertibility).
* Closure: $\det(AB) = \det(A)\det(B) \neq 0$, so the product is invertible.

$GL(n,\mathbb{R})$ is the "ambient" group; every other matrix group below is a subgroup of it.

**Example in $GL(2,\mathbb{R})$:**

$$
\begin{bmatrix} 2 & 1 \\ 1 & 1 \end{bmatrix}, \quad \det = 2(1) - 1(1) = 1 \neq 0\ \Rightarrow\ \text{invertible.}
$$


#### 2.4.2 Special linear group $SL(n, \mathbb{R})$

The subgroup of $GL(n, \mathbb{R})$ where the determinant equals exactly $1$:

$$
SL(n, \mathbb{R}) = \{ A \in GL(n,\mathbb{R}) \;:\; \det A = 1 \}.
$$

* Closure: $\det(AB) = 1 \cdot 1 = 1$.
* Inverse: $\det(A^{-1}) = 1/\det(A) = 1$.

Geometrically, $SL(n)$ preserves *signed volumes*: a transformation in $SL(n)$ may stretch, shear, or rotate, but cannot scale overall volume.

**Example in $SL(2, \mathbb{R})$:** $\begin{bmatrix} 1 & 1 \\ 1 & 2 \end{bmatrix}$ has $\det = 1$.


#### 2.4.3 Affine group $\mathrm{Aff}(n, \mathbb{R})$

An **affine transformation** is a linear transformation followed by a translation:

$$
T(\mathbf{v}) = A\mathbf{v} + \mathbf{b},\qquad A \in GL(n,\mathbb{R}),\; \mathbf{b} \in \mathbb{R}^n.
$$

The set of all such $T$ forms the **affine group** $\mathrm{Aff}(n,\mathbb{R})$ under composition. It includes scaling, rotation, translation, shear, and reflection, and — unlike linear maps — does *not* fix the origin.

**Matrix representation via homogeneous coordinates.** Embed points $\mathbf{v} \in \mathbb{R}^n$ as $\begin{bmatrix} \mathbf{v} \\ 1 \end{bmatrix} \in \mathbb{R}^{n+1}$. Then an affine map is matrix multiplication:

$$
\begin{bmatrix} A & \mathbf{b} \\ \mathbf{0}^T & 1 \end{bmatrix}
\begin{bmatrix} \mathbf{v} \\ 1 \end{bmatrix}
=
\begin{bmatrix} A\mathbf{v} + \mathbf{b} \\ 1 \end{bmatrix}.
$$

So:

$$
\mathrm{Aff}(n, \mathbb{R}) = \left\{ \begin{bmatrix} A & \mathbf{b} \\ \mathbf{0}^T & 1 \end{bmatrix} \;:\; A \in GL(n,\mathbb{R}),\; \mathbf{b} \in \mathbb{R}^n \right\} \subset GL(n+1, \mathbb{R}).
$$

This homogeneous-coordinate trick is exactly what we will use for $SE(2)$ and $SE(3)$ — those are the *rigid* affine groups, where $A$ is constrained to be a rotation.


#### 2.4.4 Orthogonal group $O(n)$

The set of $n\times n$ real matrices satisfying $A^T A = A A^T = I$:

$$
O(n) = \{ A \in \mathbb{R}^{n\times n} \;:\; A^T A = I \}.
$$

These matrices preserve the Euclidean inner product (and therefore lengths and angles): for any $\mathbf{x}, \mathbf{y}$, $\langle A\mathbf{x}, A\mathbf{y}\rangle = \langle \mathbf{x}, \mathbf{y}\rangle$.

**Determinant constraint.** From $\det(AA^T) = \det(A)^2 = \det(I) = 1$, we get $\det(A) = \pm 1$. Thus $O(n)$ splits into two disconnected pieces:

* $O(n)^+ = \{ A : \det A = +1 \}$ — pure rotations.
* $O(n)^- = \{ A : \det A = -1 \}$ — rotations composed with a reflection.

The two are disjoint ($O(n)^+ \cap O(n)^- = \emptyset$). Reflections are excluded in the next group.

**Example in $O(2)$:**

$$
R(\theta) = \begin{bmatrix} \cos\theta & -\sin\theta \\ \sin\theta & \cos\theta \end{bmatrix} \in O(2),\quad \det R = 1.
$$


#### 2.4.5 Special orthogonal group $SO(n)$

The "rotations only" subgroup of $O(n)$:

$$
SO(n) = \{ A \in O(n) \;:\; \det A = +1 \}.
$$

These matrices preserve lengths, angles, *and* orientation — pure rotations in $\mathbb{R}^n$, no reflections.

* **$SO(2)$**: rotations in 2D, parameterised by a single angle. **Abelian.**
* **$SO(3)$**: rotations in 3D, three degrees of freedom. **Not abelian** — rotations in 3D do not commute in general.

These are the most important Lie groups in robotics. We spend most of the rest of this notebook on them.

**Isometry recap.** A function $f$ is an *isometry* if $\| f(\mathbf{x}) - f(\mathbf{y}) \| = \| \mathbf{x} - \mathbf{y} \|$ for all $\mathbf{x}, \mathbf{y} \in \mathbb{R}^n$. Every $SO(n)$ element is an isometry that fixes the origin. To allow translations as well, we move to the next group.


#### 2.4.6 Special Euclidean group $SE(n)$

The group of orientation-preserving isometries of $\mathbb{R}^n$ — i.e., rigid-body motions:

$$
SE(n) = \left\{ \begin{bmatrix} R & \mathbf{t} \\ \mathbf{0}^T & 1 \end{bmatrix} \;:\; R \in SO(n),\; \mathbf{t} \in \mathbb{R}^n \right\}.
$$

Every element is a rotation $R$ followed by a translation $\mathbf{t}$, packaged into an $(n+1)\times(n+1)$ matrix in homogeneous coordinates. The group operation is matrix multiplication, which composes rigid motions correctly:

$$
\begin{bmatrix} R_1 & \mathbf{t}_1 \\ 0 & 1 \end{bmatrix}
\begin{bmatrix} R_2 & \mathbf{t}_2 \\ 0 & 1 \end{bmatrix}
=
\begin{bmatrix} R_1 R_2 & R_1 \mathbf{t}_2 + \mathbf{t}_1 \\ 0 & 1 \end{bmatrix}.
$$

The two cases used in robotics:

* **$SE(2)$** — 2D rigid motion. Three DoF ($x, y, \theta$). Used in planar SLAM.
* **$SE(3)$** — 3D rigid motion. Six DoF (three translation, three rotation). Used in full 3D SLAM, robot kinematics, computer vision.

The inverse has a clean closed form:

$$
\begin{bmatrix} R & \mathbf{t} \\ 0 & 1 \end{bmatrix}^{-1}
=
\begin{bmatrix} R^T & -R^T\mathbf{t} \\ 0 & 1 \end{bmatrix}.
$$


## 3. Manifolds and Tangent Spaces

So far we have treated matrix groups as algebraic objects. To do calculus on them — take derivatives, integrate motion, propagate uncertainty — we need a *geometric* description as well. That description is **a smooth manifold**.


### 3.1 What is a manifold?

A **manifold** is a topological space where every point has a neighbourhood that *looks like* an open subset of Euclidean space — even if the global structure is curved or twisted.

**Simplified.** Globally the shape can be complex (curvy, twisted, looped). Locally, around each point, it looks flat — like a piece of paper.

**Examples.**

1. **Surface of a ball ($S^2$).** An ant walking on a basketball feels a flat surface, but stepping back you see the curvature. The surface is a 2-dimensional manifold embedded in $\mathbb{R}^3$.
2. **Möbius strip.** Take a paper strip, give it a half-twist, glue the ends. Locally flat, but with a global twist (and only one side).
3. **Swiss roll.** A flat sheet rolled into a spiral. Each point sits on a flat tangent plane, but the global shape is curved.

Manifolds are the right setting for "spaces where calculus makes sense." Crucially, **every Lie group is a manifold** (this is part of the definition).


### 3.2 Tangent space and tangent vector

To do calculus on a manifold $M$, we need a notion of "direction at a point." This is the **tangent space** at a point $x \in M$, denoted $T_xM$.

**Construction.** Let $\gamma(t)$ be a smooth curve in $M$ with $\gamma(0) = x$. Its velocity vector $\dot\gamma(0)$ is a **tangent vector** at $x$. The **tangent space** $T_xM$ is the set of *all* such velocity vectors at $x$, taken over all smooth curves through $x$.

**Properties.**

* $T_xM$ is a vector space (you can add velocity vectors and scale them).
* $\dim T_xM = \dim M$ (a 2-manifold has a 2-dimensional tangent space at each point).

For a curved 2-manifold like a sphere, the tangent space at a point is the flat plane tangent to the sphere at that point — every direction you can "set off in" without immediately leaving the surface.

For a Lie group, the tangent space at the **identity** has special structure: it is closed under a bilinear bracket operation, and is called the **Lie algebra** of the group. We get to that in §5.


### 3.3 Matrix groups as geometric objects

A matrix group is an *algebraic* object, but we can also view it as a *geometric* one — because it sits inside Euclidean space:

$$
\mathcal{G} \subset GL_n(\mathbb{R}) \subset M_n(\mathbb{R}) \cong \mathbb{R}^{n^2}.
$$

Concretely, a $2\times 2$ matrix

$$
\begin{bmatrix} a & b \\ c & d \end{bmatrix}
$$

can be flattened into a point $\begin{bmatrix} a \\ b \\ c \\ d \end{bmatrix} \in \mathbb{R}^4$. So a $2\times 2$ matrix group lives in $\mathbb{R}^4$ as a subset.

This embedding is what lets us talk about the *tangent space* of a matrix group: it's the tangent space (in the manifold sense above) of $\mathcal{G}$ as a subset of $\mathbb{R}^{n^2}$. We compute it explicitly in the next chapter.


## 4. Lie Groups

A **Lie group** is a group that is *also* a smooth manifold, with the property that the group operations (multiplication and inversion) are smooth functions on the manifold. This combination of algebra and geometry is exactly what we want for rigid-body motion.


### 4.1 Definition: group + smooth manifold

A **Lie group** is a structure that is simultaneously:

* a **group** (in the sense of §2.1), and
* a **smooth manifold** (in the sense of §3.1),

with the additional requirement that the group multiplication $(g, h) \mapsto g \cdot h$ and inversion $g \mapsto g^{-1}$ are *smooth* maps with respect to the manifold structure.

This is more than the sum of its parts. Smoothness lets us *differentiate* group operations. That is what makes calculus on Lie groups possible — and what gives us tools like the exponential map (§6), the adjoint (§7.3), and group Jacobians (§8).

**Recap of the four ingredients.**

* **Group:** a set with a binary operation, identity, inverse, associativity (§2.1).
* **Topological space:** a set with a notion of nearness (open sets, neighbourhoods).
* **Continuous group:** group + topological space, with continuous operations.
* **Differentiable manifold:** locally Euclidean, calculus works.

A Lie group has all four: it is a continuous group whose underlying topological space is a smooth manifold and whose operations are smooth.

**Examples.** Every matrix group we listed in §2.4 is a Lie group: $GL(n)$, $SL(n)$, $O(n)$, $SO(n)$, $SE(n)$, the affine group, $\mathrm{Sim}(n)$, etc.


### 4.2 The "rigidity" of Lie groups: deriving $\mathfrak{so}(3)$ from $R^TR = I$

A central calculation: differentiating the rotation-matrix constraint reveals the *Lie algebra* — the tangent space at the identity. Let's do this for $SO(3)$.

Consider an arbitrary rotation matrix $R$ that depends smoothly on time, $R(t)$. Since it stays in $SO(3)$, it must satisfy $R(t)\,R(t)^T = I$ for all $t$. Differentiating both sides:

$$
\dot R(t)\, R(t)^T + R(t)\, \dot R(t)^T = 0.
$$

Rearranging:

$$
\dot R(t)\, R(t)^T = - \big(\dot R(t)\, R(t)^T\big)^T.
$$

A matrix that equals the negative of its own transpose is **skew-symmetric**. So **$\dot R\, R^T$ is skew-symmetric** — for *every* smooth curve $R(t)$ through any rotation matrix.

There is a one-to-one correspondence between $3 \times 3$ skew-symmetric matrices and 3D vectors. Define the hat / wedge operator:

$$
\mathbf{a}^\wedge \;=\; [\mathbf{a}]_\times \;=\; \begin{bmatrix}
0 & -a_3 & a_2 \\
a_3 & 0 & -a_1 \\
-a_2 & a_1 & 0
\end{bmatrix},
\qquad (\mathbf{a}^\wedge)^\vee = \mathbf{a}.
$$

Then there is a unique 3D vector $\boldsymbol{\phi}(t)$ with

$$
\dot R(t)\, R(t)^T = \boldsymbol{\phi}(t)^\wedge.
$$

Multiplying both sides on the right by $R(t)$ gives the **kinematic equation of $SO(3)$**:

$$
\boxed{\;\dot R(t) = \boldsymbol{\phi}(t)^\wedge\, R(t).\;}
$$

Setting $R = I$ (the identity), $\dot R(0) = \boldsymbol{\phi}(0)^\wedge$ is itself skew-symmetric. So:

> **The tangent space of $SO(3)$ at the identity is the set of $3\times 3$ skew-symmetric matrices.**

This tangent space is called $\mathfrak{so}(3)$, the **Lie algebra** of $SO(3)$. We formalise this in §5.

The same construction, applied to other matrix groups, gives their Lie algebras. We will tabulate them in §5.6.


### 4.3 Lie groups used in robotics — overview map

Before drilling into derivations, here is the map of which Lie groups appear where. We construct each in detail later.

| Group | What it represents | Where it shows up |
|---|---|---|
| $S^1 \cong SO(2)$ | 2D rotations | planar pose, heading angle |
| $SO(3)$ | 3D rotations | camera orientation, IMU attitude |
| $SE(2)$ | 2D rigid motion (rotation + translation) | planar SLAM, mobile robots |
| $SE(3)$ | 3D rigid motion | full 3D SLAM, robot arms |
| $\mathrm{Sim}(2),\ \mathrm{Sim}(3)$ | rigid motion + uniform scale | monocular SLAM (unknown scale) |
| $SE_2(3)$ | rigid motion + velocity | IMU pre-integration / inertial SLAM |
| $S^3 \cong $ unit quaternions | 3D rotations (double cover) | numerical rotation representation |

A graphical summary of these manifolds and their topologies:


## 5. Lie Algebras

The **Lie algebra** $\mathfrak{g}$ of a Lie group $G$ is the tangent space at the identity, equipped with a binary operation called the **Lie bracket**. It captures the *infinitesimal* behaviour of the group — and almost all numerical work happens in the algebra rather than in the group itself, because the algebra is a vector space (much friendlier).


### 5.1 Abstract definition

A **Lie algebra** is a triple $(\mathbb{V}, \mathbb{F}, [\cdot, \cdot])$ consisting of a vector space $\mathbb{V}$ over a field $\mathbb{F}$ (typically $\mathbb{R}$ or $\mathbb{C}$) and a binary operation $[\cdot, \cdot]: \mathbb{V}\times\mathbb{V}\to\mathbb{V}$ called the **Lie bracket**, satisfying:

1. **Closure** — $\forall \mathbf{X}, \mathbf{Y} \in \mathbb{V}: [\mathbf{X}, \mathbf{Y}] \in \mathbb{V}$.
2. **Bilinearity** — for all $\mathbf{X}, \mathbf{Y}, \mathbf{Z} \in \mathbb{V}$ and scalars $a, b \in \mathbb{F}$:
   $$
   [a\mathbf{X} + b\mathbf{Y}, \mathbf{Z}] = a[\mathbf{X}, \mathbf{Z}] + b[\mathbf{Y}, \mathbf{Z}], \quad
   [\mathbf{Z}, a\mathbf{X} + b\mathbf{Y}] = a[\mathbf{Z}, \mathbf{X}] + b[\mathbf{Z}, \mathbf{Y}].
   $$
3. **Alternativity** — $[\mathbf{X}, \mathbf{X}] = \mathbf{0}$ for every $\mathbf{X}$. Combined with bilinearity this forces $[\mathbf{X}, \mathbf{Y}] = -[\mathbf{Y}, \mathbf{X}]$ (antisymmetry).
4. **Jacobi identity** — for all $\mathbf{X}, \mathbf{Y}, \mathbf{Z}$:
   $$
   [\mathbf{X}, [\mathbf{Y}, \mathbf{Z}]] + [\mathbf{Y}, [\mathbf{Z}, \mathbf{X}]] + [\mathbf{Z}, [\mathbf{X}, \mathbf{Y}]] = \mathbf{0}.
   $$

The Lie bracket is *not* required to be associative; the Jacobi identity is the substitute. Compared to the group multiplication, the Lie bracket measures the *non-commutativity* of the corresponding group elements ("how much does $g_1 g_2$ differ from $g_2 g_1$ infinitesimally?").

**Concrete example.** The cross product on $\mathbb{R}^3$ — $(\mathbf{a}, \mathbf{b}) \mapsto \mathbf{a} \times \mathbf{b}$ — is a Lie bracket. So $(\mathbb{R}^3, \mathbb{R}, \times)$ is a Lie algebra. We will see in §5.5 that this is exactly $\mathfrak{so}(3)$ in disguise.


### 5.2 Lie algebra of a matrix group

For a matrix group $\mathcal{G} \subset GL_n(\mathbb{R})$, the Lie algebra is the **tangent space at the identity**:

$$
\mathfrak{g} = \mathfrak{g}(\mathcal{G}) = T_e \mathcal{G},
$$

where $e = I_n$ is the group identity. We chose the identity because every group has one — but it turns out the tangent space at any other point is just a translated copy of $T_e \mathcal{G}$ (left- or right-translation by a group element), so the tangent space at the identity captures everything.

Concretely, for matrix groups: an element of $\mathfrak{g}$ is the time-derivative $\dot\gamma(0)$ of any smooth curve $\gamma(t)$ through the identity, i.e., $\gamma(0) = I$. As we computed in §4.2 for $SO(3)$, this turns out to be the set of skew-symmetric matrices — but the construction is general.

**Lie bracket for matrix groups.** When the group operation is matrix multiplication, the natural bracket on the tangent space is the **commutator**:

$$
[A, B] = AB - BA.
$$

You can check: this satisfies bilinearity, alternativity ($[A,A] = 0$), and the Jacobi identity. It vanishes iff $A$ and $B$ commute — which is why the Lie bracket measures non-commutativity of the corresponding flows.


### 5.3 Generators

Consider a Lie group $G$ represented in $\mathbb{R}^{n\times n}$, with $k$ degrees of freedom. The Lie algebra $\mathfrak{g}$ is a $k$-dimensional vector space. Pick a basis $\{G_1, \dots, G_k\}$; these are the **generators** of the algebra.

Elements of $\mathfrak{g}$ are matrices in $\mathbb{R}^{n\times n}$, but as a *vector space* they are added and scaled (not multiplied). Any algebra element can be written as a linear combination of generators:

$$
\text{alg}: \mathbb{R}^k \to \mathfrak{g} \subset \mathbb{R}^{n\times n},
\qquad
\text{alg}(\mathbf{c}) = \sum_{i=1}^k c_i\, G_i.
$$

A tangent vector is a $k$-vector of *coefficients* $\mathbf{c}$, but its "matrix realisation" is an $n\times n$ matrix. Switching freely between the two is a recurring move.

We will write the generators of $\mathfrak{so}(3)$ and $\mathfrak{se}(2)$ explicitly in §5.6.


### 5.4 Hat ($\wedge$) and Vee ($\vee$) notation

To switch between the *vector* form $\mathbf{w} \in \mathbb{R}^k$ and the *matrix* form (an element of $\mathfrak{g} \subset \mathbb{R}^{n\times n}$) we use two operators:

* **Hat / wedge** $\wedge$: turns a vector into the matrix realisation in $\mathfrak{g}$.
* **Vee** $\vee$: the inverse — extracts the vector from a matrix.

For $\mathfrak{so}(3)$ (3D angular velocities), with $\mathbf{w} = [w_1,\ w_2,\ w_3]^T \in \mathbb{R}^3$:

$$
\mathbf{w}^\wedge = \begin{bmatrix}
0 & -w_3 & w_2 \\
w_3 & 0 & -w_1 \\
-w_2 & w_1 & 0
\end{bmatrix} \in \mathfrak{so}(3),
\qquad
\big(\mathbf{w}^\wedge\big)^\vee = \mathbf{w}.
$$

For $\mathfrak{se}(3)$ (6D twists $\boldsymbol{\xi} = [\boldsymbol{\rho}; \boldsymbol{\phi}] \in \mathbb{R}^6$ with $\boldsymbol{\rho}$ = translational part, $\boldsymbol{\phi}$ = rotational part):

$$
\boldsymbol{\xi}^\wedge = \begin{bmatrix} \boldsymbol{\phi}^\wedge & \boldsymbol{\rho} \\ \mathbf{0}^T & 0 \end{bmatrix} \in \mathfrak{se}(3) \subset \mathbb{R}^{4\times 4}.
$$

The bottom-right block is $0$, *not* $1$ as in the group element — that distinguishes algebra elements from group elements at a glance.

Some references (e.g., Barfoot, Forster) use $\widehat{\boldsymbol{\phi}}$ instead of $\boldsymbol{\phi}^\wedge$. Same meaning.


### 5.5 Cross product as Lie bracket on $\mathfrak{so}(3)$

The standard basis vectors of $\mathbb{R}^3$ satisfy $\mathbf{e}_1 \times \mathbf{e}_2 = \mathbf{e}_3$ (and cyclic permutations). Their hat-images — the generators of $\mathfrak{so}(3)$ — satisfy the analogous bracket relations:

$$
G_1 = \mathbf{e}_1^\wedge = \begin{pmatrix} 0 & 0 & 0 \\ 0 & 0 & -1 \\ 0 & 1 & 0 \end{pmatrix},\quad
G_2 = \mathbf{e}_2^\wedge = \begin{pmatrix} 0 & 0 & 1 \\ 0 & 0 & 0 \\ -1 & 0 & 0 \end{pmatrix},\quad
G_3 = \mathbf{e}_3^\wedge = \begin{pmatrix} 0 & -1 & 0 \\ 1 & 0 & 0 \\ 0 & 0 & 0 \end{pmatrix}.
$$

(Sign conventions vary across references; the structure constants are what matters.)

Their commutators reproduce the cross-product structure:

$$
[G_1, G_2] = G_1 G_2 - G_2 G_1 = G_3, \quad [G_2, G_3] = G_1, \quad [G_3, G_1] = G_2,
$$

which are exactly the cross-product identities in matrix form. So:

$$
\boxed{\;[\mathbf{a}^\wedge, \mathbf{b}^\wedge] = (\mathbf{a} \times \mathbf{b})^\wedge\;}
$$

In words: **the Lie bracket on $\mathfrak{so}(3)$ *is* the cross product on $\mathbb{R}^3$**, transported via $\wedge$.

This is one of the cleanest correspondences in robotics — you already know the Lie bracket on $\mathfrak{so}(3)$ if you know the cross product.


### 5.6 Examples: $\mathfrak{so}(2),\ \mathfrak{so}(3),\ \mathfrak{se}(2),\ \mathfrak{se}(3)$

The four Lie algebras we use most. Each is the tangent space at the identity of its corresponding group, derived by differentiating a constraint as in §4.2.


#### 5.6.1 $\mathfrak{so}(2)$ — 2D rotations

The Lie algebra of $SO(2)$ is the space of $2\times 2$ skew-symmetric matrices. One degree of freedom (the rotation rate $\theta$):

$$
\mathfrak{so}(2) = \left\{ [\theta]_\times = \begin{bmatrix} 0 & -\theta \\ \theta & 0 \end{bmatrix} \;:\; \theta \in \mathbb{R} \right\}.
$$

The single generator is

$$
G_1 = \begin{pmatrix} 0 & -1 \\ 1 & 0 \end{pmatrix},
\qquad [\theta]_\times = \theta\, G_1.
$$

$\mathfrak{so}(2)$ is **abelian** — its Lie bracket is identically zero — because $SO(2)$ is abelian.


#### 5.6.2 $\mathfrak{so}(3)$ — 3D rotations

The Lie algebra of $SO(3)$ is the space of $3\times 3$ skew-symmetric matrices, parameterised by 3D vectors via the hat operator (§5.4):

$$
\mathfrak{so}(3) = \left\{ \boldsymbol{\phi}^\wedge \;:\; \boldsymbol{\phi} \in \mathbb{R}^3 \right\}.
$$

A vector $\boldsymbol{\phi}$ in $\mathfrak{so}(3)$ encodes a 3D rotation in **angle-axis form**: its direction is the rotation axis and its magnitude is the rotation angle. The exponential map (next chapter) sends $\boldsymbol{\phi}^\wedge$ to the rotation matrix.

Its **Lie bracket** is the commutator on matrices, equivalently the cross product in vector form (§5.5):

$$
[\boldsymbol{\phi}_1, \boldsymbol{\phi}_2] = \big(\boldsymbol{\phi}_1^\wedge \boldsymbol{\phi}_2^\wedge - \boldsymbol{\phi}_2^\wedge \boldsymbol{\phi}_1^\wedge\big)^\vee = \boldsymbol{\phi}_1 \times \boldsymbol{\phi}_2.
$$

The relationship to $SO(3)$ is the **exponential map**:

$$
R = \exp(\boldsymbol{\phi}^\wedge) \in SO(3).
$$

We compute this exponential explicitly in §6.4 and recover **Rodrigues' formula**.


#### 5.6.3 $\mathfrak{se}(2)$ — 2D rigid motion

The Lie algebra of $SE(2)$ is 3-dimensional, parameterised by translation $(x, y)$ and rotation rate $\theta$. The matrix realisation is:

$$
\mathfrak{se}(2) = \left\{ \boldsymbol{\xi}^\wedge = \begin{bmatrix} 0 & -\theta & x \\ \theta & 0 & y \\ 0 & 0 & 0 \end{bmatrix} \;:\; \boldsymbol{\xi} = (x, y, \theta)^T \in \mathbb{R}^3 \right\}.
$$

The three generators are:

$$
G_1 = \begin{pmatrix} 0 & 0 & 1 \\ 0 & 0 & 0 \\ 0 & 0 & 0 \end{pmatrix},\quad
G_2 = \begin{pmatrix} 0 & 0 & 0 \\ 0 & 0 & 1 \\ 0 & 0 & 0 \end{pmatrix},\quad
G_3 = \begin{pmatrix} 0 & -1 & 0 \\ 1 & 0 & 0 \\ 0 & 0 & 0 \end{pmatrix}.
$$

$G_1, G_2$ generate translations along the $x$- and $y$-axes; $G_3$ generates rotation. In coefficients, $\boldsymbol{\xi}^\wedge = x G_1 + y G_2 + \theta G_3$.


#### 5.6.4 $\mathfrak{se}(3)$ — 3D rigid motion

The Lie algebra of $SE(3)$ is 6-dimensional. We split a tangent vector $\boldsymbol{\xi} \in \mathbb{R}^6$ into a translational part $\boldsymbol{\rho} \in \mathbb{R}^3$ and a rotational part $\boldsymbol{\phi} \in \mathbb{R}^3$:

$$
\boldsymbol{\xi} = \begin{bmatrix} \boldsymbol{\rho} \\ \boldsymbol{\phi} \end{bmatrix} \in \mathbb{R}^6,
\qquad
\boldsymbol{\xi}^\wedge = \begin{bmatrix} \boldsymbol{\phi}^\wedge & \boldsymbol{\rho} \\ \mathbf{0}^T & 0 \end{bmatrix} \in \mathbb{R}^{4\times 4}.
$$

**Caveat.** The "translational part" $\boldsymbol{\rho}$ in the algebra is *not* the same as the translation $\mathbf{t}$ in the group — they are related by a Jacobian matrix $\mathbf{J}$ that we derive in §6.6.

The Lie bracket on $\mathfrak{se}(3)$ is the matrix commutator:

$$
[\boldsymbol{\xi}_1, \boldsymbol{\xi}_2] = \big(\boldsymbol{\xi}_1^\wedge \boldsymbol{\xi}_2^\wedge - \boldsymbol{\xi}_2^\wedge \boldsymbol{\xi}_1^\wedge\big)^\vee.
$$

In coordinates (using $\boldsymbol{\rho}_i, \boldsymbol{\phi}_i$):

$$
[\boldsymbol{\xi}_1, \boldsymbol{\xi}_2]
= \begin{bmatrix} \boldsymbol{\phi}_1 \times \boldsymbol{\rho}_2 - \boldsymbol{\phi}_2 \times \boldsymbol{\rho}_1 \\ \boldsymbol{\phi}_1 \times \boldsymbol{\phi}_2 \end{bmatrix}.
$$

Note the $\boldsymbol{\phi}_1 \times \boldsymbol{\phi}_2$ in the rotational part: it's just the $\mathfrak{so}(3)$ bracket from §5.6.2.


## 6. The Exponential and Logarithm Maps

The **exponential map** $\exp: \mathfrak{g} \to G$ takes elements of the Lie algebra (tangent vectors at the identity) to elements of the Lie group. It is the formal way to "integrate" an infinitesimal motion into a finite one — the Lie-group analogue of $\theta \mapsto e^{i\theta}$ from §1.4.

The **logarithm map** $\log: G \to \mathfrak{g}$ is the (locally-defined) inverse: it pulls a finite group element back to its tangent vector at the identity.


### 6.1 Exp: from algebra to group

For matrix Lie groups, the exponential map is just the matrix exponential from §1.3:

$$
\exp: \mathfrak{g} \to G,\qquad
\exp(\mathbf{x}) = I + \mathbf{x} + \tfrac{1}{2!}\mathbf{x}^2 + \tfrac{1}{3!}\mathbf{x}^3 + \cdots = \sum_{i=0}^\infty \frac{1}{i!}\, \mathbf{x}^i.
$$

* Always well-defined and continuous.
* Has a closed form for many matrix groups (we derive several below).
* Generally **not** injective globally — different algebra elements can map to the same group element. (Example: in $SO(3)$, both $\boldsymbol{\phi}$ and $\boldsymbol{\phi} + 2\pi \hat{\boldsymbol{\phi}}$ map to the same rotation.)

**Geometric picture.** An algebra element $\mathbf{x}$ specifies a "direction" at the identity. Walking along the manifold in that direction for unit time lands at $\exp(\mathbf{x})$. For $S^1$, $\mathbf{x} = i\theta$ and $\exp(i\theta)$ walks $\theta$ radians along the unit circle from $1$.


### 6.2 Log: from group to algebra

The **logarithmic map** is the local inverse:

$$
\log: G \to \mathfrak{g},\qquad \exp(\log(X)) = X.
$$

Properties:

* Well-defined and smooth in a neighbourhood of the identity.
* **Not** continuous everywhere — for compact subgroups (like rotations) the algebra is unbounded but the group is bounded, so $\log$ has discontinuities at antipodal points.
* For matrix groups, can be computed by a Mercator series:
  $$
  \log(X) = \sum_{n=1}^\infty \frac{(-1)^{n-1}}{n}(X - I)^n,
  $$
  but this only converges for $X$ near $I$. In practice we use closed forms, derived per group below.

For each Lie group of interest we now derive the closed-form $\exp$ and $\log$.


### 6.3 Closed form for $SO(2)$

We want to evaluate $\exp([\theta]_\times)$ where

$$
[\theta]_\times = \begin{bmatrix} 0 & -\theta \\ \theta & 0 \end{bmatrix}.
$$

**Key observation:** $[\theta]_\times^2 = -\theta^2 I$, so the powers cycle:

$$
[\theta]_\times^2 = -\theta^2 I,\quad [\theta]_\times^3 = -\theta^2\, [\theta]_\times,\quad [\theta]_\times^4 = \theta^4 I,\quad \ldots
$$

Substitute into the Taylor series and group even/odd terms:

$$
\begin{aligned}
\exp([\theta]_\times) &= I + [\theta]_\times + \tfrac{1}{2!}[\theta]_\times^2 + \tfrac{1}{3!}[\theta]_\times^3 + \cdots \\
&= \Big(I - \tfrac{\theta^2}{2!} I + \tfrac{\theta^4}{4!} I - \cdots\Big) + \Big([\theta]_\times - \tfrac{\theta^2}{3!}[\theta]_\times + \tfrac{\theta^4}{5!}[\theta]_\times - \cdots\Big) \\
&= \cos\theta\, I + \sin\theta\, [\hat\theta]_\times \\
&= \begin{bmatrix} \cos\theta & -\sin\theta \\ \sin\theta & \cos\theta \end{bmatrix}.
\end{aligned}
$$

So:

$$
\boxed{\;\exp\Big(\begin{bmatrix} 0 & -\theta \\ \theta & 0 \end{bmatrix}\Big) = \begin{bmatrix} \cos\theta & -\sin\theta \\ \sin\theta & \cos\theta \end{bmatrix}.\;}
$$

The inverse (the **logarithm**) is just $\theta = \operatorname{atan2}(R_{21}, R_{11})$, which automatically lands in $(-\pi, \pi]$ — see [`lie_groups_se2.ipynb` §3](lie_groups_se2.ipynb) for details.

This is a *direct* analogue of Euler's formula $e^{i\theta} = \cos\theta + i\sin\theta$: $S^1 \cong SO(2)$ via the obvious isomorphism $e^{i\theta} \leftrightarrow R(\theta)$.


### 6.4 Closed form for $SO(3)$ — Rodrigues' formula

Now the 3D version. Let $\boldsymbol{\phi} = \theta\, \mathbf{n}$ where $\mathbf{n}$ is a unit vector (the rotation axis) and $\theta = \|\boldsymbol{\phi}\|$ (the rotation angle). We want

$$
R = \exp(\boldsymbol{\phi}^\wedge) = \exp(\theta\, \mathbf{n}^\wedge).
$$

**Two key identities for unit $\mathbf{n}$:**

$$
\mathbf{n}^\wedge \mathbf{n}^\wedge = \mathbf{n}\,\mathbf{n}^T - I,\qquad
\mathbf{n}^\wedge \mathbf{n}^\wedge \mathbf{n}^\wedge = -\mathbf{n}^\wedge.
$$

The first is proved by direct computation (we do this in detail below). The second follows from the first because $\mathbf{n}^\wedge \mathbf{n} = \mathbf{n} \times \mathbf{n} = \mathbf{0}$.

**Substitute into the Taylor series:**

$$
\begin{aligned}
\exp(\theta\, \mathbf{n}^\wedge)
&= I + \theta\, \mathbf{n}^\wedge + \tfrac{\theta^2}{2!}\, \mathbf{n}^\wedge\mathbf{n}^\wedge + \tfrac{\theta^3}{3!}\, \mathbf{n}^\wedge\mathbf{n}^\wedge\mathbf{n}^\wedge + \tfrac{\theta^4}{4!}\, (\mathbf{n}^\wedge)^4 + \cdots \\
&= I + \big(\theta - \tfrac{\theta^3}{3!} + \tfrac{\theta^5}{5!} - \cdots\big)\mathbf{n}^\wedge + \big(\tfrac{\theta^2}{2!} - \tfrac{\theta^4}{4!} + \cdots\big)\mathbf{n}^\wedge\mathbf{n}^\wedge \\
&= I + \sin\theta\, \mathbf{n}^\wedge + (1 - \cos\theta)\, \mathbf{n}^\wedge\mathbf{n}^\wedge \\
&= I + \sin\theta\, \mathbf{n}^\wedge + (1 - \cos\theta)\, (\mathbf{n}\,\mathbf{n}^T - I) \\
&= \cos\theta\, I + (1 - \cos\theta)\, \mathbf{n}\,\mathbf{n}^T + \sin\theta\, \mathbf{n}^\wedge.
\end{aligned}
$$

**Rodrigues' formula:**

$$
\boxed{\;R = \exp(\theta\, \mathbf{n}^\wedge) = \cos\theta\, I + (1 - \cos\theta)\, \mathbf{n}\, \mathbf{n}^T + \sin\theta\, \mathbf{n}^\wedge.\;}
$$

So $\mathfrak{so}(3)$ vectors are exactly **rotation vectors** (also called *angle-axis*), and the exponential map is Rodrigues' formula.

**Inverse direction (log map).** Take the trace of both sides:

$$
\operatorname{tr}(R) = 3\cos\theta + (1 - \cos\theta) = 1 + 2\cos\theta,
$$

so

$$
\theta = \arccos\!\left(\frac{\operatorname{tr}(R) - 1}{2}\right).
$$

For the axis: any unit eigenvector of $R$ with eigenvalue $1$ is an axis ($R\mathbf{n} = \mathbf{n}$, since rotation fixes its axis).

For $|\theta| < \pi$ this is unique. Near $\theta = \pi$ the formula $\frac{R - R^T}{2\sin\theta}$ becomes ill-conditioned and one must resort to the eigenvalue computation directly.


**Proof that $\mathbf{n}^\wedge \mathbf{n}^\wedge = \mathbf{n}\,\mathbf{n}^T - I$ for unit $\mathbf{n}$.**

The skew-symmetric matrix associated with $\mathbf{n} = (n_1, n_2, n_3)^T$ is

$$
\mathbf{n}^\wedge = \begin{bmatrix} 0 & -n_3 & n_2 \\ n_3 & 0 & -n_1 \\ -n_2 & n_1 & 0 \end{bmatrix}.
$$

Direct multiplication gives

$$
\mathbf{n}^\wedge \mathbf{n}^\wedge = \begin{bmatrix}
-n_3^2 - n_2^2 & n_1 n_2 & n_1 n_3 \\
n_1 n_2 & -n_1^2 - n_3^2 & n_2 n_3 \\
n_1 n_3 & n_2 n_3 & -n_1^2 - n_2^2
\end{bmatrix}.
$$

The outer product is

$$
\mathbf{n}\,\mathbf{n}^T = \begin{bmatrix}
n_1^2 & n_1 n_2 & n_1 n_3 \\
n_1 n_2 & n_2^2 & n_2 n_3 \\
n_1 n_3 & n_2 n_3 & n_3^2
\end{bmatrix},
$$

so

$$
\mathbf{n}\,\mathbf{n}^T - I = \begin{bmatrix}
n_1^2 - 1 & n_1 n_2 & n_1 n_3 \\
n_1 n_2 & n_2^2 - 1 & n_2 n_3 \\
n_1 n_3 & n_2 n_3 & n_3^2 - 1
\end{bmatrix}.
$$

Since $\mathbf{n}$ is a unit vector, $n_1^2 + n_2^2 + n_3^2 = 1$, so $-n_2^2 - n_3^2 = n_1^2 - 1$ (and similarly for the other diagonal entries). The off-diagonal entries already match. Hence $\mathbf{n}^\wedge \mathbf{n}^\wedge = \mathbf{n}\,\mathbf{n}^T - I$. $\square$


### 6.5 Closed form for $SE(2)$

For $\boldsymbol{\xi} = (x, y, \theta)^T \in \mathbb{R}^3$, the algebra element is

$$
\boldsymbol{\xi}^\wedge = \begin{bmatrix} 0 & -\theta & x \\ \theta & 0 & y \\ 0 & 0 & 0 \end{bmatrix}.
$$

The exponential separates into a rotation block and a translation block:

$$
\exp(\boldsymbol{\xi}^\wedge) = \begin{bmatrix} R(\theta) & V(\theta)\,\begin{bmatrix} x \\ y \end{bmatrix} \\ 0 & 1 \end{bmatrix},
$$

where

$$
R(\theta) = \begin{bmatrix} \cos\theta & -\sin\theta \\ \sin\theta & \cos\theta \end{bmatrix},
\qquad
V(\theta) = \frac{1}{\theta}\begin{bmatrix} \sin\theta & -(1-\cos\theta) \\ 1-\cos\theta & \sin\theta \end{bmatrix}.
$$

* As $\theta \to 0$, $V(\theta) \to I$ and the formula degenerates to the trivial Euclidean translation.
* The matrix $V$ converts the algebra translation $(x, y)$ into the **group** translation $V(\theta)\,(x, y)^T$. They differ by a rotation-dependent factor.

The log map inverts the closed form, recovering $(\theta, V^{-1}\mathbf{t})$.


### 6.6 Closed form for $SE(3)$

For $\boldsymbol{\xi} = \begin{bmatrix} \boldsymbol{\rho} \\ \boldsymbol{\phi} \end{bmatrix} \in \mathbb{R}^6$:

$$
\exp(\boldsymbol{\xi}^\wedge) =
\begin{bmatrix} \exp(\boldsymbol{\phi}^\wedge) & \mathbf{J}(\boldsymbol{\phi})\, \boldsymbol{\rho} \\ \mathbf{0}^T & 1 \end{bmatrix}
= \begin{bmatrix} R & \mathbf{t} \\ \mathbf{0}^T & 1 \end{bmatrix},
$$

where $R$ is the Rodrigues rotation and the **left-Jacobian** $\mathbf{J}(\boldsymbol{\phi})$ converts the algebra translation $\boldsymbol{\rho}$ into the group translation $\mathbf{t}$.

**Derivation of $\mathbf{J}$.** Write $\boldsymbol{\phi} = \theta\, \mathbf{a}$ with $\mathbf{a}$ a unit vector. Then

$$
\mathbf{J}(\boldsymbol{\phi})
= \sum_{n=0}^\infty \frac{1}{(n+1)!}\, (\boldsymbol{\phi}^\wedge)^n.
$$

Using the same $\mathbf{a}^\wedge\mathbf{a}^\wedge = \mathbf{a}\mathbf{a}^T - I$ and $\mathbf{a}^\wedge\mathbf{a}^\wedge\mathbf{a}^\wedge = -\mathbf{a}^\wedge$ identities as for Rodrigues, the series collapses to

$$
\boxed{\;\mathbf{J}(\boldsymbol{\phi}) = \frac{\sin\theta}{\theta}\, I + \Big(1 - \frac{\sin\theta}{\theta}\Big)\, \mathbf{a}\,\mathbf{a}^T + \frac{1 - \cos\theta}{\theta}\, \mathbf{a}^\wedge.\;}
$$

This is the **left-Jacobian of $SO(3)$**. Note the family resemblance to Rodrigues, but the coefficients are different — Rodrigues uses $\cos\theta$ and $\sin\theta$; $\mathbf{J}$ uses $\frac{\sin\theta}{\theta}$ and $\frac{1-\cos\theta}{\theta}$ (sinc-like terms that are well-behaved as $\theta \to 0$).

**Log map.** Given $T = \begin{bmatrix} R & \mathbf{t} \\ 0 & 1 \end{bmatrix} \in SE(3)$:

1. Recover $\boldsymbol{\phi}$ from $R$ via the $SO(3)$ log map (§6.4).
2. Compute $\mathbf{J}(\boldsymbol{\phi})$.
3. Recover $\boldsymbol{\rho} = \mathbf{J}(\boldsymbol{\phi})^{-1}\, \mathbf{t}$.

A more explicit but equivalent recipe:

1. $\theta = \|\boldsymbol{\omega}\|$ where $\boldsymbol{\omega}^\wedge = \log(R)$.
2. $R = e^{\boldsymbol{\omega}^\wedge} = I + \frac{\sin\theta}{\theta}\,\boldsymbol{\omega}^\wedge + \frac{1-\cos\theta}{\theta^2}\,(\boldsymbol{\omega}^\wedge)^2$.
3. $\mathbf{t} = \Big(I + \frac{1-\cos\theta}{\theta^2}\,\boldsymbol{\omega}^\wedge + \frac{\theta - \sin\theta}{\theta^3}\,(\boldsymbol{\omega}^\wedge)^2\Big)\, \mathbf{v}$.


### 6.7 The "Exp" capital-E shortcut

In Lie-group SLAM literature you will see *both*

$$
R = \exp([\theta]_\times) \quad\text{and}\quad R = \operatorname{Exp}(\theta).
$$

The lowercase $\exp$ takes an *algebra* element (an $n\times n$ matrix in $\mathfrak{g}$). The capital-$E$ $\operatorname{Exp}$ takes the *vector* form (the $k$-vector of coefficients) and is shorthand for "apply $\wedge$ then $\exp$":

$$
\operatorname{Exp}(\boldsymbol{\phi}) := \exp(\boldsymbol{\phi}^\wedge).
$$

Same operation, different starting representation. We use whichever is convenient.

Likewise:

$$
\operatorname{Log}(R) := \big(\log(R)\big)^\vee \in \mathbb{R}^k.
$$


## 7. Practical Operations on Lie Groups

Once you have the group, algebra, exp, and log, the practical operations needed in robotics are:

* The **action** of a group element on a point.
* The **plus and minus operators** $\oplus$ and $\ominus$ for "perturbing a pose by a vector" and "comparing two poses."
* The **adjoint representation** for transforming twists / tangent vectors between left- and right-multiplication.
* **Matrix similarity** for changing the basis in which a transformation is expressed.


### 7.1 Group action on a point

A Lie group $G$ acts on a vector space $V$ when each group element defines a transformation of $V$. For our matrix groups, the action is matrix-vector multiplication:

$$
f: SO(3) \times \mathbb{R}^3 \to \mathbb{R}^3,\quad
(R, \mathbf{p}) \mapsto f(R, \mathbf{p}) = R\,\mathbf{p}.
$$

For $SE(3)$:

$$
f: SE(3) \times \mathbb{R}^3 \to \mathbb{R}^3,\quad
(T, \mathbf{p}) \mapsto R\,\mathbf{p} + \mathbf{t},
$$

where $T = \begin{bmatrix} R & \mathbf{t} \\ 0 & 1 \end{bmatrix}$.

Group actions are bilinear in the algebra sense (we'll compute their Jacobians w.r.t. both arguments in §8).


### 7.2 The plus and minus operators ($\oplus$, $\ominus$)

Many algorithms (Kalman filters, Gauss–Newton on manifolds) want to "add a small perturbation" to a Lie group element. Plain addition does not stay in the group — $R_1 + R_2$ is generally not a rotation matrix. The right-correct operation is **right-multiplication by $\exp$ of the perturbation:**

$$
\boxed{\;X \oplus \boldsymbol{\tau} \triangleq X \cdot \exp(\boldsymbol{\tau}^\wedge) = X \cdot \operatorname{Exp}(\boldsymbol{\tau}).\;}
$$

Here $X \in G$ is a group element and $\boldsymbol{\tau} \in \mathbb{R}^k$ is a vector parameter (a "twist" for $SE(3)$, a rotation vector for $SO(3)$, etc.).

* $X \oplus \boldsymbol{\tau}$ is a proper group element (a rotation matrix, a transformation matrix, etc.).
* $\boldsymbol{\tau} = \mathbf{0}$ gives $X \oplus \mathbf{0} = X$.
* The convention $X \cdot \exp(\boldsymbol{\tau}^\wedge)$ is **right-perturbation**; some texts use **left-perturbation** $\exp(\boldsymbol{\tau}^\wedge) \cdot X$. The two differ by an adjoint (§7.3).

The inverse operation is the **minus operator** $\ominus$, used for "what's the difference between two group elements?":

$$
\boxed{\;Y \ominus X \triangleq \operatorname{Log}(X^{-1}\, Y) = \big(\log(X^{-1}\, Y)\big)^\vee.\;}
$$

* $Y \ominus X$ is a *vector* in $\mathbb{R}^k$, not a group element.
* If $Y = X$, then $Y \ominus X = \mathbf{0}$.
* Reads: "the tangent perturbation that takes you from $X$ to $Y$."

These two operators are everything you need to do gradient-based optimisation on a Lie group manifold.


### 7.3 The adjoint representation

The **adjoint** is the operator that converts a tangent vector at one point into the tangent vector at another point that produces the same group element. It answers questions like: "if I right-perturb $X$ by $\boldsymbol{\tau}$, what *left*-perturbation has the same effect?"


#### 7.3.1 Definition and key identity

Consider $X \in G$ and $\boldsymbol{\tau} \in \mathfrak{g}$ (the algebra). Right-perturbation is $X \cdot \exp(\boldsymbol{\tau})$; left-perturbation is $\exp(\boldsymbol{\sigma}) \cdot X$. We want to find the $\boldsymbol{\sigma}$ such that the two are equal:

$$
\exp(\boldsymbol{\sigma}) \cdot X = X \cdot \exp(\boldsymbol{\tau}).
$$

Right-multiplying both sides by $X^{-1}$:

$$
\exp(\boldsymbol{\sigma}) = X \cdot \exp(\boldsymbol{\tau}) \cdot X^{-1},
\qquad
\boldsymbol{\sigma} = \log(X \cdot \exp(\boldsymbol{\tau}) \cdot X^{-1}).
$$

The map $\boldsymbol{\tau} \mapsto \boldsymbol{\sigma}$ is the **adjoint of $X$**:

$$
\boxed{\;\operatorname{Ad}_X: \mathfrak{g} \to \mathfrak{g},\qquad \operatorname{Ad}_X(\boldsymbol{\tau}) = X\, \boldsymbol{\tau}\, X^{-1}.\;}
$$

The adjoint is **linear**, so as a map between vector spaces it can be represented by a matrix in $\mathbb{R}^{k\times k}$ acting on the coefficient vectors.

It preserves group composition:

$$
\operatorname{Ad}_{XY} = \operatorname{Ad}_X\, \operatorname{Ad}_Y, \qquad \operatorname{Ad}_{X^{-1}} = \operatorname{Ad}_X^{-1}.
$$

So the assignment $X \mapsto \operatorname{Ad}_X$ is a group homomorphism from $G$ into $GL(\mathfrak{g})$ — *the* group's "self-action" on its own tangent space.


#### 7.3.2 Adjoint as a between-tangent-spaces map

A useful re-phrasing: $\operatorname{Ad}_X$ converts a tangent vector at $X$ (right-coordinate) to a tangent vector at the identity (left-coordinate):

$$
y = \boldsymbol{\sigma} \oplus_{\text{left}} x = x \oplus_{\text{right}} \boldsymbol{\tau} \quad \Leftrightarrow \quad \boldsymbol{\sigma} = \operatorname{Ad}_X\, \boldsymbol{\tau}.
$$

Geometrically: $\operatorname{Ad}_X$ is the linear isomorphism $T_X M \to T_e M$ induced by left-translation by $X^{-1}$.

The **adjoint at the algebra level** ($\operatorname{ad}_x = X\, x\, X^{-1}$ linearised at $X = I$) is the matrix commutator: $\operatorname{ad}_x(y) = [x, y]$. For $\mathfrak{so}(3)$ this is the cross product (§5.5).


#### 7.3.3 Adjoint of $SE(3)$ — twist transformation

A **twist** is a 6-vector in $\mathfrak{se}(3)$ encoding linear and angular velocity:

$$
\boldsymbol{\nu} = \begin{bmatrix} \boldsymbol{\omega} \\ \mathbf{v} \end{bmatrix} \in \mathbb{R}^6.
$$

The $6\times 6$ adjoint matrix of a transformation $T = \begin{bmatrix} R & \mathbf{p} \\ 0 & 1 \end{bmatrix} \in SE(3)$ is

$$
\big[\operatorname{Ad}_T\big] = \begin{bmatrix} R & \mathbf{0} \\ \mathbf{p}^\wedge R & R \end{bmatrix} \in \mathbb{R}^{6\times 6}.
$$

This lets us convert a twist between frames:

$$
\boldsymbol{\nu}_a = \big[\operatorname{Ad}_{T_{ab}}\big]\, \boldsymbol{\nu}_b
$$

— "if you want $\boldsymbol{\nu}$ expressed in frame $a$ instead of frame $b$, multiply by the adjoint of $T_{ab}$."

For pure rotations: from $\dot R_{sb} = \boldsymbol{\omega}_s^\wedge R_{sb}$ (the kinematic equation of §4.2), so

$$
\dot R_{sb}\, R_{sb}^{-1} = \boldsymbol{\omega}_s^\wedge \in \mathfrak{so}(3),
$$

and similarly for full $SE(3)$:

$$
\dot T_{sb}\, T_{sb}^{-1} = \begin{bmatrix} \boldsymbol{\omega}_s^\wedge & \mathbf{v}_s \\ 0 & 0 \end{bmatrix} = \boldsymbol{\nu}_s^\wedge \in \mathfrak{se}(3).
$$


### 7.4 Matrix similarity and basis changes

Two square matrices $A$ and $B$ are **similar** if there exists an invertible $P$ such that

$$
B = P^{-1}\, A\, P.
$$

This is exactly what happens when you express the *same linear transformation* in two different bases — $P$ is the basis-change matrix.

Similar matrices share many properties: the same eigenvalues, characteristic polynomial, trace, determinant, rank. They are different *coordinate representations* of one underlying linear map.

In robotics this comes up whenever you change reference frames: the linear part of a rigid motion expressed in frame $b$ versus frame $e$ is related by similarity transformation (where $P$ is the rotation between the frames). And it appears as the construction inside the adjoint (§7.3), which is exactly $X \cdot (\cdot) \cdot X^{-1}$.


#### 7.4.1 Worked example: numerical similarity

Let

$$
A = \begin{pmatrix} 4 & -5 \\ 2 & -3 \end{pmatrix}, \qquad
P = \begin{pmatrix} 1 & 1 \\ 1 & 0 \end{pmatrix}.
$$

The inverse of $P$ is

$$
P^{-1} = \frac{1}{\det P}\begin{pmatrix} 0 & -1 \\ -1 & 1 \end{pmatrix} = \begin{pmatrix} 0 & 1 \\ 1 & -1 \end{pmatrix}
$$

(since $\det P = (1)(0) - (1)(1) = -1$).

Compute $AP$:

$$
AP = \begin{pmatrix} 4 & -5 \\ 2 & -3 \end{pmatrix}\begin{pmatrix} 1 & 1 \\ 1 & 0 \end{pmatrix} = \begin{pmatrix} -1 & 4 \\ -1 & 2 \end{pmatrix}.
$$

Then $B = P^{-1} A P$:

$$
B = \begin{pmatrix} 0 & 1 \\ 1 & -1 \end{pmatrix}\begin{pmatrix} -1 & 4 \\ -1 & 2 \end{pmatrix} = \begin{pmatrix} -1 & 2 \\ 0 & 2 \end{pmatrix}.
$$

So $A$ and $B = \begin{pmatrix} -1 & 2 \\ 0 & 2 \end{pmatrix}$ are similar — different entries, but the same eigenvalues ($-1$ and $2$ — read off the diagonal of $B$).


#### 7.4.2 Worked example: robotic-arm frame change

A robotic arm has two frames:

* **Base frame ($B$)** — the origin from which the arm is controlled.
* **End-effector frame ($E$)** — the gripper, used for precise manipulation.

Suppose the linear transformation in the $B$-frame is a rotation by $\theta$ followed by a translation $(x, y)$:

$$
A_B = \begin{pmatrix} \cos\theta & -\sin\theta & x \\ \sin\theta & \cos\theta & y \\ 0 & 0 & 1 \end{pmatrix}.
$$

Let $P$ be the rotation that takes coordinates in the $E$-frame to the $B$-frame (assume the origin is shared, so we only need the rotation part):

$$
P = \begin{pmatrix} \cos\phi & -\sin\phi & 0 \\ \sin\phi & \cos\phi & 0 \\ 0 & 0 & 1 \end{pmatrix}.
$$

Since $P$ is a rotation, $P^{-1} = P^T$:

$$
P^{-1} = P^T = \begin{pmatrix} \cos\phi & \sin\phi & 0 \\ -\sin\phi & \cos\phi & 0 \\ 0 & 0 & 1 \end{pmatrix}.
$$

The $E$-frame representation of the same transformation is $A_E = P^{-1}\, A_B\, P$. The arithmetic is straightforward but lengthy; the result is again a rotation+translation matrix — but now expressed in the end-effector's coordinates.

This is the same algebra you do whenever you re-express a Jacobian, screw axis, or twist in a different reference frame. The adjoint formula $\operatorname{Ad}_T(\boldsymbol{\xi}) = T\, \boldsymbol{\xi}\, T^{-1}$ from §7.3 is exactly a matrix similarity, and that is no coincidence.


## 8. Calculus on Lie Groups: Jacobians

To do gradient-based optimisation (Gauss–Newton, Levenberg–Marquardt, EKFs) on a Lie group, we need derivatives of functions $f: G \to G'$. But $G$ is a manifold, not a vector space — we cannot just write $\partial f / \partial X$. The trick: differentiate using $\oplus$ and $\ominus$ as the manifold-aware "addition" and "subtraction".


### 8.1 The Lie-group Jacobian definition

For a smooth map $f: M \to N$ between two manifolds, the Jacobian at $x \in M$ is defined by

$$
\boxed{\;
\mathbf{J} = \frac{Df(x)}{Dx} \triangleq \lim_{\boldsymbol{\tau} \to \mathbf{0}} \frac{f(x \oplus \boldsymbol{\tau}) \ominus f(x)}{\boldsymbol{\tau}} \in \mathbb{R}^{n\times m}.
\;}
$$

Here $m = \dim M$, $n = \dim N$, $\oplus$ uses $M$'s manifold structure, and $\ominus$ uses $N$'s. Both numerator and denominator are vectors in $\mathbb{R}$, so the ratio is a matrix.

Expanding the operators: $f(x \oplus \boldsymbol{\tau}) = f(x \cdot \operatorname{Exp}(\boldsymbol{\tau}))$ and $a \ominus b = \operatorname{Log}(b^{-1} \cdot a)$, so

$$
\mathbf{J} = \lim_{\boldsymbol{\tau} \to \mathbf{0}} \frac{\operatorname{Log}\!\big(f(x)^{-1}\, f(x \cdot \operatorname{Exp}(\boldsymbol{\tau}))\big)}{\boldsymbol{\tau}}.
$$

For a linear map $\boldsymbol{\tau} \mapsto \mathbf{J} \boldsymbol{\tau}$, this collapses to:

$$
\lim_{\boldsymbol{\tau} \to \mathbf{0}} \frac{\mathbf{J}\,\boldsymbol{\tau}}{\boldsymbol{\tau}} \triangleq \frac{\partial(\mathbf{J}\,\boldsymbol{\tau})}{\partial \boldsymbol{\tau}} = \mathbf{J}.
$$


### 8.2 Worked example: $\partial f(R, \mathbf{p}) / \partial R$ for $f(R, \mathbf{p}) = R\mathbf{p}$

$$
\frac{\partial f}{\partial R} = \lim_{\boldsymbol{\theta} \to \mathbf{0}} \frac{(R \oplus \boldsymbol{\theta})\,\mathbf{p} - R\,\mathbf{p}}{\boldsymbol{\theta}}
= \lim_{\boldsymbol{\theta} \to \mathbf{0}} \frac{R\, \operatorname{Exp}(\boldsymbol{\theta})\,\mathbf{p} - R\,\mathbf{p}}{\boldsymbol{\theta}}.
$$

Expand $\operatorname{Exp}(\boldsymbol{\theta}) \approx I + \boldsymbol{\theta}^\wedge$ to first order in $\boldsymbol{\theta}$:

$$
= \lim_{\boldsymbol{\theta} \to \mathbf{0}} \frac{R\,(I + \boldsymbol{\theta}^\wedge)\,\mathbf{p} - R\,\mathbf{p}}{\boldsymbol{\theta}}
= \lim_{\boldsymbol{\theta} \to \mathbf{0}} \frac{R\, \boldsymbol{\theta}^\wedge\, \mathbf{p}}{\boldsymbol{\theta}}.
$$

Use the identity $\boldsymbol{\theta}^\wedge\, \mathbf{p} = -\mathbf{p}^\wedge\, \boldsymbol{\theta}$ (the cross-product flip):

$$
= \lim_{\boldsymbol{\theta} \to \mathbf{0}} \frac{-R\, \mathbf{p}^\wedge\, \boldsymbol{\theta}}{\boldsymbol{\theta}}
= -R\, \mathbf{p}^\wedge.
$$

So:

$$
\boxed{\;\frac{\partial (R\mathbf{p})}{\partial R} = -R\, \mathbf{p}^\wedge \in \mathbb{R}^{3\times 3}.\;}
$$


### 8.3 Worked example: $\partial f(R, \mathbf{p}) / \partial \mathbf{p}$

This one is straightforward because $\mathbf{p}$ lives in the vector space $\mathbb{R}^3$:

$$
\frac{\partial (R\mathbf{p})}{\partial \mathbf{p}} = \lim_{\delta \mathbf{p} \to \mathbf{0}} \frac{R\,(\mathbf{p} + \delta \mathbf{p}) - R\,\mathbf{p}}{\delta \mathbf{p}}
= \lim_{\delta \mathbf{p} \to \mathbf{0}} \frac{R\,\delta \mathbf{p}}{\delta \mathbf{p}} = R.
$$

So:

$$
\boxed{\;\frac{\partial (R\mathbf{p})}{\partial \mathbf{p}} = R \in \mathbb{R}^{3\times 3}.\;}
$$

Combined with §8.2, the full Jacobian of the action map is $[\, -R\,\mathbf{p}^\wedge \;|\; R\, ]$.


### 8.4 Reference table of common Lie-group Jacobians

For the most common Lie-group operations, the Jacobians have been worked out and tabulated. Consult [Sola, Deray, Atchuthan — *A micro Lie theory for state estimation in robotics*](https://arxiv.org/abs/1812.01537) for the full table; the key entries are summarised below.

| Operation | Jacobian w.r.t. group | Jacobian w.r.t. vector |
|---|---|---|
| Composition $f(X, Y) = X \cdot Y$ | $\operatorname{Ad}_{Y^{-1}}$ | $I_k$ |
| Inverse $f(X) = X^{-1}$ | $-\operatorname{Ad}_X$ | n/a |
| Action $f(X, \mathbf{p}) = R\mathbf{p}$ ($SO(3)$) | $-R\,\mathbf{p}^\wedge$ | $R$ |
| Action $f(T, \mathbf{p}) = R\mathbf{p} + \mathbf{t}$ ($SE(3)$) | $\big[\, R \;\big|\; -R\,\mathbf{p}^\wedge \,\big]$ | $R$ |
| Exp $f(\boldsymbol{\tau}) = \operatorname{Exp}(\boldsymbol{\tau})$ | $\mathbf{J}_r(\boldsymbol{\tau})$ (right-Jacobian) | n/a |
| Log $f(X) = \operatorname{Log}(X)$ | $\mathbf{J}_r^{-1}(\boldsymbol{\tau})$ | n/a |

The right-Jacobian of $SO(3)$ is

$$
\mathbf{J}_r(\boldsymbol{\phi}) = I - \frac{1 - \cos\theta}{\theta^2}\,\boldsymbol{\phi}^\wedge + \frac{\theta - \sin\theta}{\theta^3}\,(\boldsymbol{\phi}^\wedge)^2,\qquad \theta = \|\boldsymbol{\phi}\|,
$$

and is what appears in IMU pre-integration covariance propagation.


## 9. Quaternions

Rotation matrices represent 3 degrees of freedom with 9 numbers — redundant. Euler angles use 3 numbers but suffer from gimbal lock. **There is no global, singularity-free parameterisation of $SO(3)$ by 3 numbers** ([Stuelpnagel 1964](https://www.malcolmdshuster.com/FC_Stuelpnagel_1964_param_SIAM.pdf)). We need 4 numbers — the **unit quaternions**.


### 9.1 Why quaternions

A quaternion is a generalisation of complex numbers. Recall that to rotate a vector in the complex plane by $\theta$ you multiply by $e^{i\theta} = \cos\theta + i\sin\theta$ — a unit complex number. **Unit quaternions do the same thing in 3D.**

In particular, **$S^3$** (the unit 3-sphere in $\mathbb{R}^4$, equivalently the unit quaternions) is a *double cover* of $SO(3)$: each rotation corresponds to two opposite quaternions $\pm \mathbf{q}$. This is why $S^3$ has the right dimension (3) but with one extra component to dodge singularities.


### 9.2 Definition

A **quaternion** has one real part and three imaginary parts, with three "imaginary units" $i, j, k$:

$$
\mathbf{q} = q_0 + q_1 i + q_2 j + q_3 k.
$$

The fundamental relations are $i^2 = j^2 = k^2 = ijk = -1$, from which $ij = k = -ji$ etc. The arithmetic of quaternions is associative but **not commutative**.

We often write a quaternion as a (scalar, vector) pair:

$$
\mathbf{q} = [s,\, \mathbf{v}]^T,\quad s = q_0 \in \mathbb{R},\quad \mathbf{v} = [q_1, q_2, q_3]^T \in \mathbb{R}^3.
$$

**Real quaternion**: imaginary part is $\mathbf{0}$ (so it's just a scalar).
**Imaginary quaternion**: real part is $0$ (so it's just a 3-vector).

A 3D point $\mathbf{p}$ is encoded as the imaginary quaternion $[0, \mathbf{p}]^T$.

For rotation we use **unit quaternions**: $\|\mathbf{q}\|^2 = q_0^2 + q_1^2 + q_2^2 + q_3^2 = 1$.


### 9.3 Quaternion multiplication as a $4\times 4$ matrix product

Given two quaternions $\mathbf{q}_1 = [s_1, \mathbf{v}_1]^T$ and $\mathbf{q}_2 = [s_2, \mathbf{v}_2]^T$, their product is:

$$
\mathbf{q}_1\, \mathbf{q}_2 = \begin{bmatrix} s_1 s_2 - \mathbf{v}_1^T \mathbf{v}_2 \\ s_1 \mathbf{v}_2 + s_2 \mathbf{v}_1 + \mathbf{v}_1 \times \mathbf{v}_2 \end{bmatrix}.
$$

Define the matrix maps:

$$
\mathbf{q}^+ = \begin{bmatrix} s & -\mathbf{v}^T \\ \mathbf{v} & sI + \mathbf{v}^\wedge \end{bmatrix},
\qquad
\mathbf{q}^\oplus = \begin{bmatrix} s & -\mathbf{v}^T \\ \mathbf{v} & sI - \mathbf{v}^\wedge \end{bmatrix}.
$$

Then:

$$
\mathbf{q}_1\, \mathbf{q}_2 = \mathbf{q}_1^+\, \mathbf{q}_2 = \mathbf{q}_2^\oplus\, \mathbf{q}_1.
$$

(Quaternion multiplication is non-commutative, so the order matters; $^+$ implements left-multiplication, $^\oplus$ right-multiplication.)


### 9.4 Conversion: quaternion → rotation matrix

The action of a unit quaternion on a 3D point is the **conjugation** $\mathbf{p}' = \mathbf{q}\, \mathbf{p}\, \mathbf{q}^{-1}$. Using the matrix forms:

$$
\mathbf{p}' = \mathbf{q}\, \mathbf{p}\, \mathbf{q}^{-1} = \mathbf{q}^+\, \mathbf{p}^+\, \mathbf{q}^{-1} = \mathbf{q}^+\, (\mathbf{q}^{-1})^\oplus\, \mathbf{p}.
$$

Computing $\mathbf{q}^+\, (\mathbf{q}^{-1})^\oplus$:

$$
\begin{bmatrix} s & -\mathbf{v}^T \\ \mathbf{v} & sI + \mathbf{v}^\wedge \end{bmatrix}
\begin{bmatrix} s & \mathbf{v}^T \\ -\mathbf{v} & sI + \mathbf{v}^\wedge \end{bmatrix}
= \begin{bmatrix} 1 & \mathbf{0}^T \\ \mathbf{0} & \mathbf{v}\,\mathbf{v}^T + s^2 I + 2s\,\mathbf{v}^\wedge + (\mathbf{v}^\wedge)^2 \end{bmatrix}.
$$

The bottom-right $3\times 3$ block *is* the rotation matrix corresponding to the quaternion:

$$
\boxed{\;R = \mathbf{v}\,\mathbf{v}^T + s^2 I + 2s\,\mathbf{v}^\wedge + (\mathbf{v}^\wedge)^2.\;}
$$


### 9.5 Conversion: quaternion → rotation vector (axis–angle)

Take the trace of the formula in §9.4:

$$
\operatorname{tr}(R) = \operatorname{tr}(\mathbf{v}\mathbf{v}^T) + 3s^2 + 0 + \operatorname{tr}((\mathbf{v}^\wedge)^2)
= \|\mathbf{v}\|^2 + 3s^2 - 2\|\mathbf{v}\|^2 = 4s^2 - 1.
$$

Combine with $\theta = \arccos\!\big(\frac{\operatorname{tr}(R) - 1}{2}\big)$ from §6.4:

$$
\theta = \arccos(2s^2 - 1).
$$

This simplifies (using $\cos\theta = 2\cos^2\frac{\theta}{2} - 1$ and $\cos\frac{\theta}{2} = s$) to:

$$
\boxed{\;\theta = 2\,\arccos s.\;}
$$

**Rotation axis.** The imaginary part of $\mathbf{q}$ is unchanged under the rotation, so the unit axis is just $\mathbf{v} / \sin\frac{\theta}{2}$. From the Pythagorean identity, $\sin\frac{\theta}{2} = \sqrt{1 - s^2}$, so:

$$
\boxed{\;
\theta = 2\arccos(q_0),\qquad
[n_x, n_y, n_z]^T = \frac{[q_1, q_2, q_3]^T}{\sin\frac{\theta}{2}} = \frac{[q_1, q_2, q_3]^T}{\sqrt{1 - q_0^2}}.
\;}
$$


## 10. Generalisations beyond $SE(3)$

Two extensions of $SE(3)$ that show up in practice.


### 10.1 $\mathrm{Sim}(2)$ and $\mathrm{Sim}(3)$ — motion + scale

The **similarity groups** add a uniform-scale component to rigid motion:

$$
\mathrm{Sim}(n) = \{ s R + \mathbf{t} \;:\; s > 0,\ R \in SO(n),\ \mathbf{t} \in \mathbb{R}^n \}.
$$

In matrix form, $\mathrm{Sim}(n)$ elements live in $GL(n+1, \mathbb{R})$:

$$
T = \begin{bmatrix} s R & \mathbf{t} \\ 0 & 1 \end{bmatrix} \in \mathbb{R}^{(n+1) \times (n+1)}.
$$

Its Lie algebra $\mathfrak{sim}(n)$ adds one extra dimension (the scale rate $\kappa$) on top of $\mathfrak{se}(n)$:

$$
\mathfrak{sim}(2)\ni\xi = \begin{bmatrix} \kappa & -\theta & v_x \\ \theta & \kappa & v_y \\ 0 & 0 & 0 \end{bmatrix},
$$

so $\dim \mathrm{Sim}(2) = 4$ (translation 2 + rotation 1 + scale 1) and $\dim \mathrm{Sim}(3) = 7$ (translation 3 + rotation 3 + scale 1).

**Where it shows up.** Monocular SLAM cannot recover absolute scale — a video of a small toy car is indistinguishable from a video of a real car. Optimising over $\mathrm{Sim}(3)$ acknowledges this gauge by leaving scale as a free parameter; loop closures provide constraints that pin it down up to a global factor.


### 10.2 $SE_2(3)$ — motion + velocity (IMU SLAM)

For inertial / IMU-aided SLAM, we want to bundle position, orientation, *and velocity* as one Lie-group state. The result is **$SE_2(3)$**, an extension of $SE(3)$ where each element is a $5 \times 5$ matrix:

$$
T = \begin{bmatrix} R & \mathbf{v} & \mathbf{p} \\ \mathbf{0}^T & 1 & 0 \\ \mathbf{0}^T & 0 & 1 \end{bmatrix} \in SE_2(3),
$$

with

* $R \in SO(3)$ — orientation,
* $\mathbf{v} \in \mathbb{R}^3$ — velocity,
* $\mathbf{p} \in \mathbb{R}^3$ — position.

The Lie algebra $\mathfrak{se}_2(3)$ is 9-dimensional ($\boldsymbol{\omega} \in \mathfrak{so}(3)$, plus rates of velocity and position):

$$
\boldsymbol{\xi}^\wedge = \begin{bmatrix} \boldsymbol{\omega}^\wedge & \mathbf{a} & \mathbf{b} \\ 0 & 0 & 0 \\ 0 & 0 & 0 \end{bmatrix},
\qquad
\boldsymbol{\xi} = [\boldsymbol{\omega};\, \mathbf{a};\, \mathbf{b}]^T \in \mathbb{R}^9.
$$

The $5 \times 5$ formulation has a beautiful property: when you propagate the state forward by an IMU pre-integration step, the **gravity-corrected** dynamics are *linear* in $SE_2(3)$ matrix form. This is the basis of "right-invariant EKF on $SE_2(3)$" for inertial SLAM (Barrau & Bonnabel, "The Invariant Extended Kalman Filter as a Stable Observer", 2017).

**Applications.** Tightly-coupled visual–inertial SLAM (VINS-Mono, ORB-SLAM3 in inertial mode), pre-integrated IMU factors in factor graphs.


## 11. Applications in Robotics

We close with several concrete uses of the Lie-group machinery: angular velocity, screws and twists, robot pose representation, integration over $SO(3)$, and an EKF on $SE(3)$.


### 11.1 Angular velocity, linear velocity

If a frame $\{b\}$ rotates with respect to a spatial frame $\{s\}$ at rate $\dot\theta$ around a unit axis $\hat{\boldsymbol{\omega}}$, the **angular-velocity vector** in the spatial frame is

$$
\boldsymbol{\omega}_s = \hat{\boldsymbol{\omega}}\,\dot\theta \in \mathbb{R}^3.
$$

The corresponding skew-symmetric matrix in $\mathfrak{so}(3)$ describes the rate of change of the rotation matrix:

$$
\boxed{\;\dot R_{sb} = \boldsymbol{\omega}_s^\wedge\, R_{sb}\;}
\qquad \Leftrightarrow \qquad
\dot R_{sb}\, R_{sb}^{-1} = \boldsymbol{\omega}_s^\wedge \in \mathfrak{so}(3).
$$

This is exactly the kinematic equation we derived in §4.2.

**Linear velocity.** A point on a rotating rigid body at radius $r$ moves with linear speed

$$
v = \frac{s}{t} = \frac{r\,\theta}{t} = r\,\omega.
$$

For a body undergoing combined translation and rotation, this combines additively (we make this precise via twists in §11.2).


### 11.2 Screws and twists

A **screw** is an axis with a pitch — it captures the geometry of "rotate about an axis while sliding along it":

$$
s = \begin{bmatrix} s_\omega \\ s_v \end{bmatrix} = \begin{bmatrix} \text{angular velocity at }\dot\theta = 1 \\ \text{linear velocity of the origin at }\dot\theta = 1 \end{bmatrix}.
$$

The linear velocity of the origin has two contributions:

* $h\,\hat{s}$ — translation along the axis (pitch $h$);
* $-\hat{s} \times \mathbf{q}$ — translation due to the axis being offset from the origin by $\mathbf{q}$.

A **twist** is the full $\mathfrak{se}(3)$ vector, a screw scaled by an angular rate $\dot\theta$:

$$
\boldsymbol{\nu} = \begin{bmatrix} \boldsymbol{\omega} \\ \mathbf{v} \end{bmatrix}_{6\times 1} = s\, \dot\theta.
$$

The kinematic equation generalises from $SO(3)$ to $SE(3)$:

$$
\dot T_{sb}\, T_{sb}^{-1} = \begin{bmatrix} \boldsymbol{\omega}_s^\wedge & \mathbf{v}_s \\ 0 & 0 \end{bmatrix} = \boldsymbol{\nu}_s^\wedge \in \mathfrak{se}(3).
$$

The **adjoint** transforms a twist between frames (§7.3.3):

$$
\boldsymbol{\nu}_a = \big[\operatorname{Ad}_{T_{ab}}\big]\, \boldsymbol{\nu}_b
\quad\text{where}\quad
\big[\operatorname{Ad}_T\big] = \begin{bmatrix} R & \mathbf{0} \\ \mathbf{p}^\wedge R & R \end{bmatrix}.
$$

This is heavily used in robot kinematics — chaining velocities through a serial manipulator.


### 11.3 Representing robot pose

A pose has both a position and an orientation. Choices:

| Representation | Pros | Cons |
|---|---|---|
| Rotation matrix + translation $(R, \mathbf{t})$ | direct group element, exact composition | 9+3 = 12 numbers, 6 redundancy |
| $SE(3)$ homogeneous matrix | composition is matrix mult | 16 numbers |
| Quaternion + translation $(\mathbf{q}, \mathbf{t})$ | 4+3 = 7, no singularity | unit-norm constraint, double cover |
| Euler angles + translation | minimal (6 numbers) | gimbal lock, ambiguous conventions |
| Twist $\boldsymbol{\xi} \in \mathfrak{se}(3)$ + reference frame | minimal | needs $\exp$ to recover pose |
| Dual quaternion | algebraically clean | unfamiliar arithmetic |

The right choice depends on what you do with the pose. For *storage* and *exchange*: quaternion + translation, or homogeneous matrix. For *optimisation*: minimal $\mathfrak{se}(3)$ tangent vector with manifold updates. See [Furgale 2014, "Representing Robot Pose: The Good, the Bad, and the Ugly"](https://web.archive.org/web/20161029231029/https://paulfurgale.info/news/2014/6/9/representing-robot-pose-the-good-the-bad-and-the-ugly) for an opinionated overview.


### 11.4 Continuous integration over $SO(3)$

To average a function over all rotations, you need to integrate over the rotation group:

$$
I = \int_{SO(3)} f(R)\, dR.
$$

**Common techniques:**

* **Monte Carlo.** Sample uniformly from $SO(3)$ (e.g., random quaternions on the 3-sphere, normalised). Average $f$ over samples. Easy and asymptotically correct.
* **Quadrature.** Specialised quadrature rules for $SO(3)$ (Euler-angle products, $S^3$-spherical-harmonic rules) give deterministic integrals with bounded error.
* **Symmetry exploitation.** When $f$ depends only on the angle a vector makes with a fixed direction (or some other rotation-invariant quantity), the integral often collapses to a 1D integral.

**Example.** Compute $\int_{SO(3)} (R\,\mathbf{e}_x)_z^2\, dR / \operatorname{vol}(SO(3))$ — the average squared $z$-component of the rotated $x$-axis. By isotropy, the squared components $x^2, y^2, z^2$ of any unit vector under uniform rotation each average to $1/3$ (since $x^2 + y^2 + z^2 = 1$). So the answer is $1/3$ — no integration required.

This sort of trick is what lets analytic results in IMU pre-integration covariance survive without fancy calculus over $SO(3)$.


### 11.5 EKF on $SE(3)$ — worked example

A complete EKF for map-based localisation on $SE(3)$, with state on the manifold and tangent-space covariance.

**State.** Pose $X \sim \mathcal{N}(\bar X, \mathbf{P}) \in SE(3)$. The covariance is defined on the tangent space:

$$
\mathbf{P} = \mathbb{E}\big[ (X \ominus \bar X)\, (X \ominus \bar X)^T \big] \in \mathbb{R}^{6 \times 6}.
$$

**Beacons (known).** Fixed 3D landmarks $\mathbf{b}_k \in \mathbb{R}^3$ in the world frame.

**Motion model.** With control input $\mathbf{u}_i$ (e.g., a measured twist) and additive zero-mean noise $\mathbf{w}$:

$$
X_i = f(X_{i-1}, \mathbf{u}_i) = X_{i-1} \oplus (\mathbf{u}_i\, dt + \mathbf{w}),
\qquad \mathbf{w} \sim \mathcal{N}(\mathbf{0}, \mathbf{Q}).
$$

**Measurement model.** Each beacon is observed in the body frame:

$$
\mathbf{y}_k = h(X) = X^{-1}\, \mathbf{b}_k + \mathbf{v},
\qquad \mathbf{v} \sim \mathcal{N}(\mathbf{0}, \mathbf{R}).
$$

**Prediction step.** Propagate the mean by the motion model and the covariance by the linearised dynamics:

$$
\begin{aligned}
X &\leftarrow X \oplus \mathbf{u}_i\, dt, \\
F &= \frac{Df}{Dx}, \qquad G = \frac{Df}{D\mathbf{u}}, \\
\mathbf{P} &\leftarrow F\, \mathbf{P}\, F^T + G\, \mathbf{Q}\, G^T.
\end{aligned}
$$

**Correction step.** For each beacon $k$:

$$
\begin{aligned}
\mathbf{z}_k &= \mathbf{y}_k - \hat X^{-1}\, \mathbf{b}_k & &\text{(innovation)} \\
H &= \frac{\partial h}{\partial X} & &\text{(measurement Jacobian)} \\
\mathbf{Z}_k &= H\, \mathbf{P}\, H^T + \mathbf{R} & &\text{(innovation covariance)} \\
\mathbf{K} &= \mathbf{P}\, H^T\, \mathbf{Z}_k^{-1} & &\text{(Kalman gain)} \\
\hat X &\leftarrow \hat X \oplus \mathbf{K}\, \mathbf{z}_k & &\text{(state update via }\oplus\text{)} \\
\mathbf{P} &\leftarrow \mathbf{P} - \mathbf{K}\, \mathbf{Z}_k\, \mathbf{K}^T & &\text{(covariance update)}
\end{aligned}
$$

The crucial difference from a vector-space EKF is that the **state update uses $\oplus$** (manifold-aware update via $\exp$) rather than plain addition. The covariance lives in tangent space throughout.

This is one of the cleanest places where Lie-group machinery pays off. See [`error_state_extended_kalman_filter_vio.ipynb`](error_state_extended_kalman_filter_vio.ipynb) for a fully worked VIO example.


## 12. Reference Cheat Sheet


### 12.1 Summary table of Lie groups in robotics

| Lie group | Manifold | Size | Dim | $\mathcal{X} \in \mathcal{M}$ | Constraint | Lie algebra | Tangent | $\exp(\boldsymbol{\tau})$ | Composition | Action |
|----------|----------|------|-----|-------------------------------|------------|-------------|----------|---------------|-------------|--------|
| $n$-D vector | $\mathbb{R}^n$ | $n$ | $n$ | $\mathbf{v} \in \mathbb{R}^n$ | $\mathbf{v}-\mathbf{v}=0$ | $\mathbb{R}^n$ | $\boldsymbol{\tau} \in \mathbb{R}^n$ | $\mathbf{v}=\exp(\boldsymbol{\tau})$ | $\mathbf{v}_1+\mathbf{v}_2$ | $\mathbf{v}+\mathbf{x}$ |
| Circle | $S^1$ | $2$ | $1$ | $z \in \mathbb{C}$ | $z^*z = 1$ | $i\theta$ | $\theta \in \mathbb{R}$ | $z=\exp(i\theta)$ | $z_1 z_2$ | $z\,x$ |
| 2D Rotation | $SO(2)$ | $4$ | $1$ | $R \in \mathbb{R}^{2\times2}$ | $R^TR = I$ | $[\theta]_\times$ | $\theta \in \mathbb{R}$ | $R=\exp([\theta]_\times)$ | $R_1R_2$ | $R\,\mathbf{x}$ |
| 2D rigid motion | $SE(2)$ | $9$ | $3$ | $\begin{bmatrix}R & \mathbf{t}\\0&1\end{bmatrix}$ | $R^TR = I$ | $\mathfrak{se}(2)$ | $\boldsymbol{\tau}\in\mathbb{R}^3$ | matrix exp | $M_1M_2$ | $R\mathbf{x}+\mathbf{t}$ |
| 3-sphere | $S^3$ | $4$ | $3$ | $q\in\mathbb{H}$ | $q^*q=1$ | $\boldsymbol{\theta}/2$ | $\boldsymbol{\theta}\in\mathbb{R}^3$ | $q=\exp(\boldsymbol{\theta}/2)$ | $q_1q_2$ | $q\,x\,q^*$ |
| 3D Rotation | $SO(3)$ | $9$ | $3$ | $R\in\mathbb{R}^{3\times3}$ | $R^TR=I$ | $[\boldsymbol{\theta}]_\times$ | $\boldsymbol{\theta}\in\mathbb{R}^3$ | $R=\exp([\boldsymbol{\theta}]_\times)$ | $R_1R_2$ | $R\,\mathbf{x}$ |
| 3D rigid motion | $SE(3)$ | $16$ | $6$ | $\begin{bmatrix}R&\mathbf{t}\\0&1\end{bmatrix}$ | $R^TR=I$ | $\mathfrak{se}(3)$ | $\boldsymbol{\tau}\in\mathbb{R}^6$ | matrix exp | $M_1M_2$ | $R\mathbf{x}+\mathbf{t}$ |

**Notes.**

* $\mathrm{Sim}(2),\ \mathrm{Sim}(3)$: rigid motion + uniform scale (monocular SLAM).
* $SE_2(3)$: rigid motion + velocity (IMU pre-integration, inertial SLAM).


### 12.2 Quick reference

**$\mathfrak{so}(3)$.** $3\times 3$ skew-symmetric matrices, isomorphic to $\mathbb{R}^3$ via the hat operator:

$$
[\mathbf{x}]_\times = \begin{bmatrix} 0 & -x_3 & x_2 \\ x_3 & 0 & -x_1 \\ -x_2 & x_1 & 0 \end{bmatrix},\qquad [\mathbf{x}]_\times = -[\mathbf{x}]_\times^T.
$$

Bracket = cross product. Exp = Rodrigues.

**$SO(2)$.** $2\times 2$ orthogonal matrices with $\det = +1$:

$$
R(\theta) = \begin{bmatrix} \cos\theta & -\sin\theta \\ \sin\theta & \cos\theta \end{bmatrix}.
$$

**$SO(3)$.** $3\times 3$ orthogonal matrices with $\det = +1$. $R^TR = I$. The set of skew-symmetric matrices $\mathfrak{so}(3)$ is the tangent space at $I$.

**$SE(2)$.** $3\times 3$ matrices

$$
T = \begin{bmatrix} \cos\theta & -\sin\theta & r_x \\ \sin\theta & \cos\theta & r_y \\ 0 & 0 & 1 \end{bmatrix}.
$$

**$SE(3)$.** $4\times 4$ matrices

$$
T = \begin{bmatrix} R & \mathbf{p} \\ 0 & 1 \end{bmatrix},\quad R \in SO(3),\ \mathbf{p} \in \mathbb{R}^3,
$$

with inverse

$$
T^{-1} = \begin{bmatrix} R^T & -R^T\mathbf{p} \\ 0 & 1 \end{bmatrix}.
$$

**$\mathfrak{se}(3)$.** $4\times 4$ matrices

$$
\boldsymbol{\xi}^\wedge = \begin{bmatrix} \boldsymbol{\phi}^\wedge & \boldsymbol{\rho} \\ 0 & 0 \end{bmatrix}.
$$

**References.**

* Sola, Deray, Atchuthan — [*A micro Lie theory for state estimation in robotics*](https://arxiv.org/abs/1812.01537) (the cleanest practical reference).
* Barfoot — *State Estimation for Robotics* (Cambridge, 2017).
* Chirikjian — *Stochastic Models, Information Theory, and Lie Groups* (vols I–II).
* Strasdat — Sophus library and his thesis.
